<a href="https://colab.research.google.com/github/UO295831/Computer-Vision-project-2/blob/main/CV_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix

In [8]:
import torch

# This checks if Colab gave you a GPU. If yes, it uses 'cuda'. If no, it falls back to 'cpu'.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch is currently using: {DEVICE}")

# Let's double check exactly which graphics card you were assigned
if torch.cuda.is_available():
    print(f"Graphics Card: {torch.cuda.get_device_name(0)}")

PyTorch is currently using: cuda
Graphics Card: Tesla T4


# Globals

In [4]:
# --- Hardware ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch is currently using: {DEVICE}")
if torch.cuda.is_available():
    print(f"Graphics Card: {torch.cuda.get_device_name(0)}")

# --- Hyperparameters ---
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 20

# --- Paths ---
base_path = "/content/drive/MyDrive/CV_project/CV_Project_data/"
DATA_DIR = "/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_Subset"
DATA_DIR1 = "/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_Mini_Subset"


PyTorch is currently using: cuda
Graphics Card: Tesla T4


# Utils

In [ ]:
def plot_training_history(train_losses, val_losses, title="Training History"):
    """Plots the loss curves for your final report."""
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss', color='blue', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', color='red', linewidth=2)
    plt.title(title)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.show()

# Data

### ⚙️ One-Time Setup (Run Once & Collapse)

In [ ]:
# Downloading the file directly into the Drive folder, since is to heavy for our personal laptops
!wget https://zenodo.org/records/14963880/files/RRDataset_test.tar.gz?download=1 -P /content/drive/MyDrive/CV_project/CV_Project_data/

--2026-05-21 16:42:52--  https://zenodo.org/records/14963880/files/RRDataset_test.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20117869400 (19G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_test.tar.gz?download=1.1’

RRDataset_test.tar. 100%[===================>]  18.74G  18.4MB/s    in 41m 41s 

2026-05-21 17:24:34 (7.67 MB/s) - ‘/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_test.tar.gz?download=1.1’ saved [20117869400/20117869400]



In [3]:
# 1. DELETE THE ARCHIVE TO FREE UP SPACE
archive_path = "/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_test.tar.gz"
if os.path.exists(archive_path):
    os.remove(archive_path)
    print("Deleted the 20GB .tar.gz file to free up your Google Drive space!")

# 2. PEEK INSIDE THE NEW FOLDER
dataset_path = "/content/drive/MyDrive/CV_project/CV_Project_data/RRDataset_final"

print("\n Inside 'RRDataset_final':")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        print(f"\n📁 [MAIN FOLDER] {item}")
        # Look one level deeper to see the subfolders
        subfolders = [f for f in os.listdir(item_path) if os.path.isdir(os.path.join(item_path, f))]
        if subfolders:
            print(f"   ↳ Subfolders: {subfolders}")


🔍 Inside 'RRDataset_final':

📁 [MAIN FOLDER] transfer
   ↳ Subfolders: ['ai', 'real']

📁 [MAIN FOLDER] redigital
   ↳ Subfolders: ['ai', 'real']


In [9]:

base_path = "/content/drive/MyDrive/CV_project/CV_Project_data/"

print("Inspecting dataset\n")

# Go through the main data folder
for main_item in sorted(os.listdir(base_path)):
    main_path = os.path.join(base_path, main_item)
    if os.path.isdir(main_path):
        print(f"📁 {main_item}/")

        # Look inside each folder (e.g., RRDataset_final)
        for sub_item in sorted(os.listdir(main_path)):
            sub_path = os.path.join(main_path, sub_item)
            if os.path.isdir(sub_path):
                print(f"   ┣━ 📁 {sub_item}/")

                # Peek one more level deep to see the "ai" / "real" splits
                for deep_item in sorted(os.listdir(sub_path)):
                    deep_path = os.path.join(sub_path, deep_item)
                    if os.path.isdir(deep_path):
                        print(f"   ┃  ┣━ 📁 {deep_item}/")

Inspecting dataset

📁 RRDataset_Subset/
   ┣━ 📁 Fake_AI/
   ┃  ┣━ 📁 Original/
   ┃  ┣━ 📁 Redigitized/
   ┃  ┣━ 📁 Transmitted/
   ┣━ 📁 Real/
   ┃  ┣━ 📁 Original/
   ┃  ┣━ 📁 Redigitized/
   ┃  ┣━ 📁 Transmitted/
📁 RRDataset_final/
   ┣━ 📁 redigital/
   ┃  ┣━ 📁 ai/
   ┃  ┣━ 📁 real/
   ┣━ 📁 transfer/
   ┃  ┣━ 📁 ai/
   ┃  ┣━ 📁 real/
📁 RRDataset_original_train_val/
   ┣━ 📁 train/
   ┃  ┣━ 📁 ai/
   ┃  ┣━ 📁 real/
   ┣━ 📁 val/
   ┃  ┣━ 📁 ai/
   ┃  ┣━ 📁 real/


### 🚀 The PyTorch Data Pipeline

In [ ]:
dataset = Mydataset()

## 🧪 Empirical Test for Model Selection (Mini-VIBES)
This section evaluates ResNet-50, EfficientNet-B3, and ConvNeXt-Tiny on a 1,800-image prototype subset to determine the optimal backbone for the RRDataset.

### 1. Subsample (Prototype Generation)
Extracts 300 images per class to build a lightweight, balanced subset for rapid testing.

In [3]:
import os
import random
import shutil

# Fix the random seed
random.seed(42)

target_base = os.path.join(base_path, "RRDataset_Mini_Subset")
SAMPLES_PER_CLASS = 300

source_mappings = {
    ("Real", "Original"): os.path.join(base_path, "RRDataset_original_train_val/train/real"),
    ("Fake_AI", "Original"): os.path.join(base_path, "RRDataset_original_train_val/train/ai"),
    ("Real", "Transmitted"): os.path.join(base_path, "RRDataset_final/transfer/real"),
    ("Fake_AI", "Transmitted"): os.path.join(base_path, "RRDataset_final/transfer/ai"),
    ("Real", "Redigitized"): os.path.join(base_path, "RRDataset_final/redigital/real"),
    ("Fake_AI", "Redigitized"): os.path.join(base_path, "RRDataset_final/redigital/ai")
}

print(f"⚙️ Building the PROTOTYPE subset ({SAMPLES_PER_CLASS} images per class)...")

for (rf_class, tf_class), source_dir in source_mappings.items():
    target_dir = os.path.join(target_base, rf_class, tf_class)
    os.makedirs(target_dir, exist_ok=True)
    all_images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    selected_images = random.sample(all_images, min(SAMPLES_PER_CLASS, len(all_images)))

    copied_count = 0
    for img_name in selected_images:
        src_path = os.path.join(source_dir, img_name)
        dst_path = os.path.join(target_dir, img_name)
        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)
            copied_count += 1

    print(f"📁 Placed {len(os.listdir(target_dir))} images in [{rf_class} / {tf_class}]")

print("\n✅ Mini-Subset Generation Complete!")

⚙️ Building the PROTOTYPE subset (300 images per class)...
📁 Placed 300 images in [Real / Original]
📁 Placed 300 images in [Fake_AI / Original]
📁 Placed 300 images in [Real / Transmitted]
📁 Placed 300 images in [Fake_AI / Transmitted]
📁 Placed 300 images in [Real / Redigitized]
📁 Placed 300 images in [Fake_AI / Redigitized]

✅ Mini-Subset Generation Complete!


### 2. Preprocessing & Data Loading
Applies strict ImageNet base preprocessing (Resize to 224x224, Tensor conversion, Normalization) to ensure a fair evaluation of pre-trained weights.

In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Point to the NEW mini-subset

BATCH_SIZE = 32

# 2. Define the lightweight Dataset Class
class RRDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.rf_labels = []

        self.rf_map = {"Real": 0, "Fake_AI": 1}
        self.tf_map = {"Original": 0, "Transmitted": 1, "Redigitized": 2}

        # Scan the 1,800 image folder
        for rf_class, rf_idx in self.rf_map.items():
            for tf_class, tf_idx in self.tf_map.items():
                folder = os.path.join(root_dir, rf_class, tf_class)
                if not os.path.exists(folder): continue
                for img_name in os.listdir(folder):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append(os.path.join(folder, img_name))
                        self.rf_labels.append(rf_idx) # We only need Real/Fake for this test

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        # Apply the chosen preprocessing
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(self.rf_labels[idx], dtype=torch.long)

# 3. Define your separate Transformation Pipelines
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 4. Build the Loader
# ⚠️ CRITICAL: We pass `val_transform` here to ensure a clean, un-flipped
# scientific baseline for the Mini-VIBES model selection test!
mini_dataset = RRDataset(DATA_DIR1, transform=val_transform)
train_loader = DataLoader(mini_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ Preprocessing locked in! Found {len(mini_dataset)} images ready for the test.")

✅ Preprocessing locked in! Found 1800 images ready for the test.


### 3. Feature Extraction & Evaluation (Linear Probing)
Extracts raw mathematical features from an untrained 500-image sample using ResNet-50, EfficientNet-B3, and ConvNeXt-Tiny. Trains a Logistic Regression on these features to prove which architecture inherently separates Real vs. AI images best.

In [9]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

# 1. Setup Hardware
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Running VIBES empirical test on: {DEVICE}")

# 2. Download Pre-trained Backbones (ImageNet Weights)
print("\n📦 Downloading Pre-trained Backbones...")
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
effnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

# Strip classification heads to turn them into pure Feature Extractors
resnet.fc = nn.Identity()
effnet.classifier = nn.Identity()
convnext.classifier = nn.Identity()

models_dict = {
    "ResNet-50": resnet,
    "EfficientNet-B3": effnet,
    "ConvNeXt-Tiny": convnext
}

# 3. Feature Extraction Function
def extract_features(model, dataloader, num_samples=500):
    model = model.to(DEVICE)
    model.eval()

    features_list = []
    labels_list = []
    samples_processed = 0

    with torch.no_grad():
        for images, rf_labels in dataloader:
            images = images.to(DEVICE)

            # Forward pass
            features = model(images)

            # Global pooling for CNN feature maps
            if features.ndim == 4:
                features = torch.mean(features, dim=[2, 3])

            features_list.append(features.cpu().numpy())
            labels_list.append(rf_labels.numpy())

            samples_processed += images.size(0)

            if samples_processed >= num_samples:
                break

    X = np.vstack(features_list)
    y = np.concatenate(labels_list)

    return X, y

# 4. Run the Experiment
print("\n🔬 Starting Feature Extraction (Using exactly 500 images)...")
results = {}

for name, model in models_dict.items():
    start_time = time.time()
    print(f"   -> Testing {name}...")

    # Extract features
    X, y = extract_features(model, train_loader, num_samples=500)

    # Train/Test Split just for the Logistic Regression (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train Logistic Regression WITH the StandardScaler pipeline upgrade
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42))
    clf.fit(X_train, y_train)

    # Score the model
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds) * 100

    elapsed = time.time() - start_time
    results[name] = acc
    print(f"      Done in {elapsed:.1f}s. Validation Accuracy: {acc:.2f}%")

# 5. The Grand Finale: Declare the Winner
print("\n🏆 --- MINI-VIBES LEADERBOARD --- 🏆")
sorted_results = sorted(results.items(), key=lambda item: item[1], reverse=True)
for rank, (name, acc) in enumerate(sorted_results, 1):
    print(f"#{rank}: {name:<15} | Linear Probe Accuracy: {acc:.2f}%")

print(f"\n💡 Conclusion: You should use **{sorted_results[0][0]}** as your multi-task backbone!")

⚙️ Running VIBES empirical test on: cuda

📦 Downloading Pre-trained Backbones...

🔬 Starting Feature Extraction (Using exactly 500 images)...
   -> Testing ResNet-50...
      Done in 14.7s. Validation Accuracy: 84.47%
   -> Testing EfficientNet-B3...
      Done in 15.1s. Validation Accuracy: 79.61%
   -> Testing ConvNeXt-Tiny...
      Done in 17.7s. Validation Accuracy: 87.38%

🏆 --- MINI-VIBES LEADERBOARD --- 🏆
#1: ConvNeXt-Tiny   | Linear Probe Accuracy: 87.38%
#2: ResNet-50       | Linear Probe Accuracy: 84.47%
#3: EfficientNet-B3 | Linear Probe Accuracy: 79.61%

💡 Conclusion: You should use **ConvNeXt-Tiny** as your multi-task backbone!


# Network

In [ ]:
model = Mymodel()

# Train

In [ ]:
Model.train()

# Test

In [ ]:
Model.eval()